# Flugzeug fliegen: einen PID-Regler einstellen

In diesem Notebook probieren Sie aus, wie sich ein **PID-Regler** auf die Flughöhe eines Flugzeugs auswirkt.

**Ihr Ziel:** Das Flugzeug soll möglichst gut der eingezeichneten Sollhöhe folgen. Die Sollhöhe ist die Höhe, die das Flugzeug erreichen soll.

Führen Sie die Zellen der Reihe nach aus. Wenn Sie eine Einstellung ändern, wird der Flug erneut simuliert.


In [ ]:
# Benötigte Bibliotheken laden (beim ersten Mal kann die Installation etwas dauern)
%pip install -q ctairplane
import control as ct
import ctairplane as ap


## Das Flugzeugmodell

Das Flugzeug reagiert nicht sofort auf die Stellgröße: Je nachdem, wie Sie das Höhenruder betätigen, steigt oder sinkt es mit einer gewissen Verzögerung. Die Art und Weise, wie das System auf einen Eingang reagiert wird in der Kybernetik „Dynamik“ genannt. Es gibt viele verschiedene Möglichkeiten, eine Dynamik mathematisch zu beschreiben z.B. durch Differentialgleichungen, Zustandsraummodelle oder Übertragungsfunktionen. Sie müssen das hier verwendete Modell weder verstehen noch verändern, um den PID-Regler auszuprobieren.

Die folgende Zelle richtet dieses Modell ein.


In [ ]:
# Internes Flugzeugmodell – für die Übung müssen Sie diesen Code nicht ändern.
plant = ct.tf(
    [-0.15, 1],
    [0.2, 1.24, 1.2, 1]
)


## Die Sollhöhe

Die **Sollhöhe** gibt vor, auf welcher Höhe das Flugzeug zu einem bestimmten Zeitpunkt fliegen soll. In der Grafik ist sie als Linie dargestellt:

- **Die ersten fünf Sekunden:** Die Sollhöhe bleibt bei null.
- **Waagerechte Abschnitte:** Die gewünschte Höhe bleibt gleich.
- **Schräge Abschnitte:** Das Flugzeug soll steigen oder sinken.

In [ ]:
reference = ap.waypoints([
    (0, 0),
    (5, 0),       # Anfang: Höhe halten
    (10, 4),      # langsam auf eine höhere Sollhöhe steigen
    (22, 4),      # Höhe halten
    (30, -3),     # sinken
    (42, -3),     # niedrigere Höhe halten
    (50, 3),      # wieder steigen
    (62, 3),      # Höhe halten
    (68, 0),      # zurück zur Ausgangshöhe
    (73, 0),
])


## Ablauf und Bewertung

Der Flug dauert **73 Sekunden**. Das Höhenruder kann nur begrenzt ausschlagen: Die Steuerung wird auf Werte zwischen **−5 und +5** begrenzt. Wenn der Regler einen noch stärkeren Ausschlag verlangt, wird trotzdem nur der jeweilige Grenzwert verwendet.

Bewertet wird, wie nah das Flugzeug im Durchschnitt an der Sollhöhe ist. Je geringer die durchschnittliche Abweichung, desto besser:

- unter **0,6 m** ein Punkt
- unter **0,3 m** zwei Punkte
- unter **0,2 m** drei Punkte
- unter **0,06 m** vier Punkte

 Die vom Team gewählten Reglerparameter müssen bis zum **25.10.2026, 23:59 Uhr** per E-Mail an Tristan Schneider (tristan.schneider@ist.uni-stuttgart.de) gesendet werden.

In [ ]:
task = ap.Task(plant, reference, duration=73, u_max=5, tol=0.5)


## Den PID-Regler einstellen

Ein PID-Regler stellt das Höhenruder automatisch ein. Er vergleicht fortlaufend die **Sollhöhe** mit der **tatsächlichen Höhe** und reagiert auf den Unterschied.

Mit den drei Schiebereglern stellst du ein, wie der Regler reagiert:

- **P – proportionale Reaktion:** Je größer der P-Wert, desto kräftiger reagiert der Regler, wenn das Flugzeug von der Sollhöhe abweicht. Ein zu hoher Wert kann heftige Korrekturen und Überschwingen verursachen.
- **I – anhaltende Abweichung:** Der I-Anteil merkt sich, wie lange und wie stark das Flugzeug bisher von der Sollhöhe abgewichen ist. So kann er einen Fehler ausgleichen, der trotz der anderen Einstellungen bestehen bleibt. Solche Fehler können zum Beispiel durch Unterschiede des realen Systems zum Modell auftreten. Ein zu hoher Wert kann Schwingungen verursachen oder dazu führen, dass das Höhenruder nach einer starken Korrektur zu lange in die falsche Richtung arbeitet.
- **D – Dämpfung:** Der D-Anteil berücksichtigt, wie schnell sich die Höhe gerade verändert. Er kann helfen, das Flugzeug abzubremsen, bevor es über die Sollhöhe hinausschießt. Ein zu hoher Wert kann die Regelung ebenfalls verschlechtern.

Es gibt nicht für jeden Abschnitt eine perfekte Einstellung. Finden Sie Einstellungen, die über den **gesamten Flug** gut funktionieren.


In [ ]:
ap.tune_pid(task, kp=0, ki=0, kd=0)


## Probieren Sie es systematisch aus

Das nachfolgende Vorgehen ist eine mögliche Herangehensweise, geeignete Reglerparameter zu bestimmen.

Ändern Sie zunächst immer nur **einen** Schieberegler. Beobachten Sie nach jedem Versuch die Flugbahn und die Bewertung.

1. **P untersuchen:** Setzen Sie I und D zunächst auf null. Erhöhen Sie P schrittweise. Wird die Reaktion schneller? Wann beginnt das Flugzeug zu überschwingen?
2. **D ergänzen:** Erhöhen Sie D langsam. Wird das Überschwingen geringer? Was passiert, wenn D sehr groß wird?
3. **I ergänzen:** Stellen Sie I zunächst auf null und beobachten Sie, ob das Flugzeug auf den längeren waagerechten Abschnitten einen kleinen Höhenfehler behält. Erhöhen Sie I schrittweise. Wird dieser Fehler kleiner? Was passiert bei sehr großem I?
4. **Gemeinsam abstimmen:** Verändern Sie anschließend P, I und D gemeinsam in kleinen Schritten. Versuchen Sie, eine gute Bewertung zu erreichen, ohne dass das Flugzeug stark schwingt oder das Höhenruder ständig an seiner Grenze liegt.

**Tipp:** Notieren Sie sich gute Einstellungen. Ein höherer Wert ist nicht automatisch besser – entscheidend ist, wie gut die Kombination aus allen drei Werten über den gesamten Flug funktioniert.
